# 14 — Fidelity & Trace Distance

Turn the inside of the sphere into the **Bloch ball**, and pick up two honest yardsticks for "how different are two states?": the **trace distance**, which for a qubit is half the straight-line distance between Bloch vectors and tells you the best odds of telling two states apart in one shot, and the **fidelity**, which extends the inner product of `01/05` to mixed states — with its convention fixed once and for all for the rest of the course.

**Prerequisites:** `01/12_from_states_to_density_matrix` ($\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$, eigenvalues of a Hermitian matrix), `01/13_pure_vs_mixed` (mixtures live inside the sphere; $|+\rangle$ vs $I/2$; the $X$ measurement that tells them apart), `01/05_inner_product_and_dirac` (the Bhattacharyya $\leftrightarrow$ inner product row, which promised fidelity today).

**What you'll be able to do**
- Locate pure and mixed states in the Bloch ball — surface, interior, centre — and read mixedness as depth.
- Derive, for a qubit, $T(\rho, \sigma) = \tfrac12|\vec r - \vec s|$ and its operational meaning: the best one-shot guess between $\rho$ and $\sigma$ succeeds with probability $\tfrac12(1 + T)$ — and check it on $|+\rangle$ vs $I/2$ with the $X$ measurement of `01/13`.
- State the fidelity with the convention this course uses — $F(\rho,\sigma) = \big(\mathrm{tr}\sqrt{\sqrt\rho\,\sigma\sqrt\rho}\big)^2$, equal to $|\langle\varphi|\psi\rangle|^2$ for pure states — compute it by hand in the cases that matter, and know how it relates to the trace distance (they are *not* one number in two disguises).

In `01/03` every pure state sat on the surface of the Bloch sphere; `01/13` showed mixtures fill the inside. Today we make that picture quantitative, with two distances you will reach for again and again — in `01/15` on real noisy data, in `02/12` where fidelity becomes the Bures distance, and in module 04 where quantum optimal transport measures distances between states its own way.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum import density
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state
from qot_course.quantum.gates import PAULI_X, PAULI_Y, PAULI_Z

rng = np.random.default_rng(0)   # random states for the scatter plot and the exercises
viz.use_course_style()

def outer(psi):
    """|psi><psi| by hand (01/12)."""
    return np.outer(psi, psi.conj())

def bloch_of(rho):
    """Bloch vector of any qubit rho (01/12, 01/13)."""
    return np.array([np.trace(rho @ P).real for P in (PAULI_X, PAULI_Y, PAULI_Z)])

## 1. The Bloch ball

`01/13` established the geometry: $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ is a valid state exactly when $|\vec r| \le 1$ (eigenvalues $\tfrac12(1 \pm |\vec r|) \ge 0$), pure states are the surface $|\vec r| = 1$, mixtures are strictly inside, $I/2$ is the centre, and purity is $\tfrac12(1 + |\vec r|^2)$. So the state space of a qubit is a solid **ball**, and "how mixed" is "how deep". Let's place three familiar states — $|+\rangle$, the half-mixed state $\tfrac12|+\rangle\langle+| + \tfrac12\,I/2$ of `01/13`, and $I/2$ — on the ball, and on a cross-section through it that we will reuse for the distances.

In [ ]:
rho_plus, rho_zero = outer(KET_PLUS), outer(KET_0)
rho_mixed = np.eye(2) / 2
half_mixed = 0.5 * rho_plus + 0.5 * rho_mixed       # halfway from |+> to the centre (01/13 section 3, t = 1/2)

for name, r in [("|+>", rho_plus), ("half-mixed", half_mixed), ("I/2", rho_mixed)]:
    rv = bloch_of(r)
    print(f"{name:11s} Bloch vector {np.round(rv, 3)}   length {np.linalg.norm(rv):.3f}   purity {density.purity(r):.3f}")

viz.plot_bloch(rho_plus, title="pure |+>  (on the surface)")
viz.plot_bloch(half_mixed, title="half-mixed  (inside)")
viz.plot_bloch(rho_mixed, title="I/2  (at the centre)")
plt.show()

**Read the output and the figures.** The three arrows point the same way ($+x$) and shrink: length $1$ on the surface, $\tfrac12$ halfway in, $0$ at the centre, with purity $1$, $0.625$, $\tfrac12$ — the numbers of `01/13` §3, now as positions. "How deep inside the ball" *is* "how mixed"; the geometry you learned for pure states holds the mixed ones too, and a noisy device's shortened Bloch vector (`01/15`) will be a point in this ball.

## 2. Trace distance: half the distance between Bloch vectors — and the best one-shot guess

How far apart are two states? The first honest answer is the **trace distance**

$$ T(\rho, \sigma) = \tfrac12\,\|\rho - \sigma\|_1 = \tfrac12 \sum_k |\mu_k|, $$

where $\mu_k$ are the eigenvalues of the Hermitian matrix $\rho - \sigma$ (the *trace norm* is the sum of the absolute eigenvalues). For a qubit you can compute it by hand. Write both states in Bloch form: $\rho - \sigma = \tfrac12(\vec r - \vec s)\cdot\vec\sigma$ — the identity parts cancel. `01/13` (exercise 4) showed that $\vec v\cdot\vec\sigma$ has eigenvalues $\pm|\vec v|$, so $\rho - \sigma$ has eigenvalues $\pm\tfrac12|\vec r - \vec s|$, and

$$ T(\rho, \sigma) = \tfrac12\big(\tfrac12|\vec r - \vec s| + \tfrac12|\vec r - \vec s|\big) = \tfrac12\,|\vec r - \vec s| . $$

**Trace distance is half the Euclidean distance between the two points in the ball.** It ranges from $0$ (same state) to $1$ (antipodes, $|\vec r - \vec s| = 2$ — orthogonal pure states), and it is a genuine metric: symmetric, zero only for equal states, triangle inequality.

Why this particular norm? Because of what it *means*. Suppose someone hands you one copy of a qubit, prepared as $\rho$ or as $\sigma$ with equal odds, and you get one measurement to guess which. The best success probability over all measurements (the **Helstrom bound**) is

$$ P_{\text{success}} = \tfrac12\big(1 + T(\rho, \sigma)\big), $$

achieved by measuring in the eigenbasis of $\rho - \sigma$ and guessing $\rho$ on the positive-eigenvalue outcome. It is one line: measuring a basis $\{|b_0\rangle, |b_1\rangle\}$ and guessing $\rho$ on $b_0$, $\sigma$ on $b_1$, you win with probability $P = \tfrac12\langle b_0|\rho|b_0\rangle + \tfrac12\langle b_1|\sigma|b_1\rangle = \tfrac12 + \tfrac12\langle b_0|(\rho - \sigma)|b_0\rangle$ (because $\langle b_1|\sigma|b_1\rangle = 1 - \langle b_0|\sigma|b_0\rangle$). The bracket of a Hermitian matrix is at most its largest eigenvalue — here $+\tfrac12|\vec r - \vec s| = T$ — with equality when $|b_0\rangle$ is that eigenvector. So $P \le \tfrac12(1 + T)$, and the eigenbasis reaches it. $T = 0$: a coin toss; $T = 1$: never wrong. Now the payoff from `01/13`: for $|+\rangle$ vs $I/2$, $\vec r = (1,0,0)$, $\vec s = 0$, so $T = \tfrac12$ and the best one-shot guess succeeds $75\%$ of the time — and the $X$ measurement of `01/13` §1 (*the* eigenbasis of $\rho - \sigma = \tfrac12 X$) should achieve exactly that. Let's check all of it.

In [ ]:
def trace_distance_by_hand(rho, sigma):
    """(1/2) sum |eigenvalues of rho - sigma|."""
    return 0.5 * np.abs(np.linalg.eigvalsh(rho - sigma)).sum()

def guess_success(rho, sigma, basis):
    """One-shot guessing game: measure in `basis` = (|b0>, |b1>), guess rho on b0, sigma on b1; each prepared with prob 1/2."""
    b0, b1 = basis
    return 0.5 * np.vdot(b0, rho @ b0).real + 0.5 * np.vdot(b1, sigma @ b1).real

def guess_success_identity(rho, sigma, b0):
    """The same number written as 1/2 + 1/2 <b0|(rho - sigma)|b0> -- the one-line derivation of the Helstrom bound."""
    return 0.5 + 0.5 * np.vdot(b0, (rho - sigma) @ b0).real

states = {"|0>": rho_zero, "|+>": rho_plus, "half-mixed": half_mixed, "I/2": rho_mixed}
print(f"{'pair':24s} {'T by hand':>10s} {'|r-s|/2':>9s} {'library':>9s}   best one-shot guess (1+T)/2")
names = list(states)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        T = trace_distance_by_hand(states[a], states[b])
        half_dist = 0.5 * np.linalg.norm(bloch_of(states[a]) - bloch_of(states[b]))
        print(f"{a + ' vs ' + b:24s} {T:10.4f} {half_dist:9.4f} {density.trace_distance(states[a], states[b]):9.4f}   {(1 + T) / 2:.4f}")

# |+> vs I/2: the X measurement of 01/13 achieves the Helstrom bound; the Z measurement does not
print("\n|+> vs I/2, T = 0.5, Helstrom bound (1+T)/2 =", 0.75)
print("  guess with the X measurement (eigenbasis of rho - sigma = X/2):", round(guess_success(rho_plus, rho_mixed, (KET_PLUS, KET_MINUS)), 4), "  = 1/2 + 1/2 <+|(rho - sigma)|+> =", round(guess_success_identity(rho_plus, rho_mixed, KET_PLUS), 4))
print("  guess with the Z measurement                                  :", round(guess_success(rho_plus, rho_mixed, (KET_0, KET_1)), 4), "  (a coin toss: same diagonal)")

# |0> vs |+>: the optimal basis is the eigenbasis of rho - sigma, tilted halfway between the two arrows
diff = rho_zero - rho_plus
evals, evecs = np.linalg.eigh(diff)
opt_basis = (evecs[:, 1], evecs[:, 0])                    # positive eigenvalue -> guess |0>
print("\n|0> vs |+>: eigenvalues of rho - sigma", np.round(evals, 4), " -> T =", round(trace_distance_by_hand(rho_zero, rho_plus), 4))
print("  optimal measurement axis (Bloch vector of the + eigenvector):", np.round(bloch_of(outer(evecs[:, 1])), 3), " = (r - s)/|r - s| =", np.round((bloch_of(rho_zero) - bloch_of(rho_plus)) / np.sqrt(2), 3))
print("  success with that measurement:", round(guess_success(rho_zero, rho_plus, opt_basis), 4), "  bound (1+T)/2 =", round((1 + trace_distance_by_hand(rho_zero, rho_plus)) / 2, 4))
print("  success measuring Z instead   :", round(guess_success(rho_zero, rho_plus, (KET_0, KET_1)), 4), "  (guess |0> on 0, |+> on 1)")

**Read the output.** The hand formula $\tfrac12\sum|\mu_k|$, the geometric $\tfrac12|\vec r - \vec s|$ and the library agree for every pair. Two pure states at right angles on the sphere, $|0\rangle$ and $|+\rangle$, are $T = \tfrac{\sqrt2}{2} \approx 0.707$ apart; $|+\rangle$ and $I/2$ are $\tfrac12$ apart; the half-mixed state sits exactly halfway. And the meaning holds: for $|+\rangle$ vs $I/2$ the $X$ measurement of `01/13` wins the guessing game $75\%$ of the time — the Helstrom bound to the digit — while the $Z$ measurement, blind to the difference, is a coin toss. For $|0\rangle$ vs $|+\rangle$ the optimal axis is the bisector $(\vec r - \vec s)/|\vec r - \vec s|$ of the two arrows, and it reaches $\tfrac12(1 + 0.707) \approx 0.854$ where $Z$ alone gets $0.75$. A distance you can *play a game to measure* — that is what makes the trace distance honest.

## 3. Fidelity: the inner product, extended to mixed states — convention fixed

The second yardstick runs the other way: $1$ for identical states, $0$ for perfectly distinguishable ones. For two *pure* states you already own it: `01/05` read $|\langle\varphi|\psi\rangle|^2$ as the yes-probability of the question "are you $|\varphi\rangle$?", asked of $|\psi\rangle$, and promised it would be called **fidelity** today. In Bloch terms, `01/05` §5 gave $|\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$: $1$ for the same point, $\tfrac12$ at right angles, $0$ for antipodes.

The general definition — the **Uhlmann–Jozsa fidelity** — extends it to any two density matrices:

$$ F(\rho, \sigma) = \Big(\mathrm{tr}\sqrt{\sqrt{\rho}\,\sigma\,\sqrt{\rho}}\Big)^2 . $$

It looks forbidding, but it reduces by hand in every case you will meet this module:

- **Both pure**, $\rho = |\psi\rangle\langle\psi|$, $\sigma = |\varphi\rangle\langle\varphi|$: a projector is its own square root, so $\sqrt\rho\,\sigma\sqrt\rho = |\psi\rangle\langle\psi|\varphi\rangle\langle\varphi|\psi\rangle\langle\psi| = |\langle\varphi|\psi\rangle|^2\,|\psi\rangle\langle\psi|$, whose square root is $|\langle\varphi|\psi\rangle|\,|\psi\rangle\langle\psi|$ with trace $|\langle\varphi|\psi\rangle|$. So $F = |\langle\varphi|\psi\rangle|^2$ — the `01/05` number, as promised.
- **One pure**, $\rho = |\psi\rangle\langle\psi|$, $\sigma$ anything: the same steps give $F = \langle\psi|\sigma|\psi\rangle$ — the Born probability (`01/12` §3) that $\sigma$ answers "yes, I am $|\psi\rangle$". For $|+\rangle$ vs $I/2$: $\langle+|\tfrac I2|+\rangle = \tfrac12$.
- **Both diagonal** in the same basis, $\rho = \mathrm{diag}(p)$, $\sigma = \mathrm{diag}(q)$: everything commutes and $F = \big(\sum_x\sqrt{p_x q_x}\big)^2$ — the square of the Bhattacharyya coefficient of the `01/05` dictionary row. Classical overlap is the diagonal case of fidelity.
- **Any two qubits:** there is a closed form, $F = \mathrm{tr}(\rho\sigma) + 2\sqrt{\det\rho\,\det\sigma}$ (Hübner 1992), which exercise 4 checks.

**A convention to fix, once.** Textbooks disagree on whether "fidelity" means this squared quantity or its square root $\sqrt F = \mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho}$ (Nielsen & Chuang use the root). **This course uses the squared $F$ above** — so that $F = |\langle\varphi|\psi\rangle|^2$ is a probability and `density.fidelity` returns it — and writes the root as the **Uhlmann fidelity** $F_U = \sqrt F$ when it is needed: the Bures distance of `02/12`, $d_B = \sqrt{2(1 - F_U)}$, and its returns in `03/13` and `05/05` use $F_U$. When you read a fidelity anywhere, check which one it is; here, it is the probability.

Fidelity and trace distance both measure closeness, and they do move oppositely — but they are **not** one number in two disguises. What is true is a pair of inequalities (Fuchs & van de Graaf 1999):

$$ 1 - \sqrt{F} \;\le\; T \;\le\; \sqrt{1 - F}, $$

with the upper bound an equality for two pure states. Let's verify the hand reductions, then draw the band.

In [ ]:
def sqrtm_psd(M):
    """Square root of a positive semidefinite Hermitian matrix via its eigen-decomposition."""
    w, V = np.linalg.eigh(M)
    return (V * np.sqrt(np.clip(w, 0, None))) @ V.conj().T

def fidelity_by_hand(rho, sigma):
    """(tr sqrt(sqrt(rho) sigma sqrt(rho)))^2 -- the definition, three lines of linear algebra."""
    s = sqrtm_psd(rho)
    return np.trace(sqrtm_psd(s @ sigma @ s)).real ** 2

F = density.fidelity
print("definition by hand vs library, |0> vs half-mixed:", round(fidelity_by_hand(rho_zero, half_mixed), 6), round(F(rho_zero, half_mixed), 6))
print("both pure   : F(|0>,|+>) =", round(F(rho_zero, rho_plus), 4), "  |<0|+>|^2 =", round(abs(np.vdot(KET_0, KET_PLUS)) ** 2, 4), "  (1 + r.s)/2 =", round((1 + bloch_of(rho_zero) @ bloch_of(rho_plus)) / 2, 4))
print("one pure    : F(|+>, I/2) =", round(F(rho_plus, rho_mixed), 4), "  <+|I/2|+> =", round(np.vdot(KET_PLUS, rho_mixed @ KET_PLUS).real, 4))
p, q = np.array([0.7, 0.3]), np.array([0.2, 0.8])
print("both diagonal: F(diag p, diag q) =", round(F(np.diag(p).astype(complex), np.diag(q).astype(complex)), 4), "  (sum sqrt(p q))^2 =", round(float(np.sum(np.sqrt(p * q)) ** 2), 4))
print("antipodes   : F(|0>,|1>) =", round(F(rho_zero, outer(KET_1)), 4), "   same state: F(|+>,|+>) =", round(F(rho_plus, rho_plus), 4))
print("\nconvention check: sqrt(F) = Uhlmann fidelity F_U = tr sqrt(sqrt(rho) sigma sqrt(rho)); for |0> vs |+>:  F =", round(F(rho_zero, rho_plus), 4), "  F_U =", round(np.sqrt(F(rho_zero, rho_plus)), 4))

# the Fuchs-van de Graaf band on 1500 random pairs of qubit states (uniform in the ball), plus the pure-pure edge
def random_state():
    """A qubit state drawn uniformly from the Bloch ball."""
    r = rng.normal(size=3)
    r /= np.linalg.norm(r)
    return density.density_from_bloch(r * rng.uniform() ** (1 / 3))
pairs = [(random_state(), random_state()) for _ in range(1500)]
Fs = np.array([F(a, b) for a, b in pairs])
Ts = np.array([density.trace_distance(a, b) for a, b in pairs])

def random_pure():
    theta, phi = rng.uniform([0, 0], [np.pi, 2 * np.pi])
    return outer(qubit_state(theta, phi))
pure_pairs = [(random_pure(), random_pure()) for _ in range(300)]
Fp = np.array([F(a, b) for a, b in pure_pairs])
Tp = np.array([density.trace_distance(a, b) for a, b in pure_pairs])
print("\nFuchs-van de Graaf holds on all 1800 pairs:", bool(np.all((1 - np.sqrt(Fs) <= Ts + 1e-9) & (Ts <= np.sqrt(1 - Fs) + 1e-9))), "   pure pairs sit on the upper edge T = sqrt(1-F):", np.allclose(Tp, np.sqrt(1 - Fp)))

grid = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(7.0, 4.6))
ax.scatter(Fs, Ts, s=8, alpha=0.35, color=COLORS["quantum"], label="1500 random pairs (mixed states)")
ax.scatter(Fp, Tp, s=8, alpha=0.6, color=COLORS["highlight"], label="300 random pairs of pure states")
ax.plot(grid, np.sqrt(1 - grid), color=COLORS["flow"], lw=2, label=r"upper bound $T = \sqrt{1-F}$ (pure states)")
ax.plot(grid, 1 - np.sqrt(grid), color=COLORS["muted"], lw=2, ls="--", label=r"lower bound $T = 1-\sqrt{F}$")
for name, (a, b) in {"|+> vs I/2": (rho_plus, rho_mixed), "|0> vs |+>": (rho_zero, rho_plus)}.items():
    ax.plot(F(a, b), density.trace_distance(a, b), "o", color=COLORS["source"], ms=9)
    ax.annotate(name, (F(a, b), density.trace_distance(a, b)), textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.set_xlabel("fidelity F (1 = same state)"); ax.set_ylabel("trace distance T (0 = same state)")
ax.set_title("Two yardsticks, one band: the Fuchs-van de Graaf inequalities", pad=12)
ax.legend(fontsize=8, loc="upper right")
plt.show()

**Read the output and the figure.** Every hand reduction matches the library: pure–pure fidelity is the `01/05` overlap $\tfrac12(1 + \vec r\cdot\vec s) = \tfrac12$ for $|0\rangle$ vs $|+\rangle$; pure–mixed is the Born probability $\langle+|\tfrac I2|+\rangle = \tfrac12$; diagonal–diagonal is the squared Bhattacharyya coefficient; antipodes give $0$ and a state with itself $1$. The scatter shows the relationship between the two yardsticks honestly: every pair lies inside the band $1 - \sqrt F \le T \le \sqrt{1 - F}$, pure pairs lie *exactly* on the upper edge (for them $T = \sqrt{1 - F}$, so the two are indeed one number), and mixed pairs spread below it — the same fidelity can go with noticeably different trace distances. High fidelity always means low trace distance and vice versa, which is why "they move oppositely" is a fair slogan — but pick the one whose *meaning* you need: $T$ for "how well can I tell them apart in one shot", $F$ for "how much does one state look like the other" (and, in `02/12`, for the geometry of the ball).

## Your turn

1. **Self and antipodes.** Compute $T$ and $F$ of $|+\rangle$ with itself and of $|0\rangle$ with $|1\rangle$, by the Bloch formulas *before* calling the library: which pair is perfectly distinguishable, and what is the Helstrom success probability for each?
2. **Distance to the centre.** Compute $T(|+\rangle, \text{half-mixed})$ and $T(|+\rangle, I/2)$ from $\tfrac12|\vec r - \vec s|$, and $F$ for both by the one-pure reduction $\langle+|\sigma|+\rangle$. Then build the general point $\rho(t) = (1-t)|+\rangle\langle+| + t\,I/2$ of `01/13` and show $T(|+\rangle, \rho(t)) = t/2$ and $F = 1 - t/2$ — where on the Fuchs–van de Graaf figure does this family sit?
3. **Play the game.** Draw two random pure states with `rng`, compute $T$, build the optimal measurement from the eigenvectors of $\rho - \sigma$, and simulate the guessing game with $N = 20000$ rounds (flip a fair coin to prepare, sample the measurement with the Born rule, guess): does the win rate land on $\tfrac12(1 + T)$ within the error bar of `01/10`? Try a deliberately bad basis and watch the rate fall.
4. **Harder: the qubit closed forms.** Prove by hand that $T = \tfrac12|\vec r - \vec s|$ (done in §2) and that for pure states $F = \tfrac12(1 + \vec r\cdot\vec s)$, then show the upper Fuchs–van de Graaf bound is an equality for pure pairs: $\sqrt{1 - F} = \sqrt{\tfrac12(1 - \vec r\cdot\vec s)} = \tfrac12|\vec r - \vec s| = T$ (use $|\vec r - \vec s|^2 = 2 - 2\vec r\cdot\vec s$ for unit vectors). Finally verify Hübner's closed form $F = \mathrm{tr}(\rho\sigma) + 2\sqrt{\det\rho\det\sigma}$ against `density.fidelity` on 500 random pairs from the ball, and explain from it why $F(\rho, I/2) = \tfrac12 + \sqrt{\det\rho}$ grows as $\rho$ gets more mixed.

## What you built

- You placed pure and mixed states in the Bloch ball and read mixedness as depth.
- You derived the trace distance of a qubit by hand, $T = \tfrac12|\vec r - \vec s|$, and gave it its meaning — the best one-shot guess succeeds with probability $\tfrac12(1 + T)$ — checking that the $X$ measurement of `01/13` achieves it for $|+\rangle$ vs $I/2$.
- You fixed the course's fidelity convention, $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2 = |\langle\varphi|\psi\rangle|^2$ for pure states (and $F_U = \sqrt F$ for the Bures distance of `02/12`), reduced it by hand in the pure, one-pure and diagonal cases, and saw the Fuchs–van de Graaf band that ties it to the trace distance.

Lovely work: the abstract idea of mixedness is now something you can *see* (a point in a ball), *measure* (two numbers with meanings), and *play a game with*.

**Dictionary so far** (the table from `01/13`, plus the row you earned today):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| marginal | partial trace |
| transport plan | quantum coupling |

(The last two rows are still forward-looking — earned in `01/16` and `04/04`.)

## What's next

Time to meet a genuinely mixed state — not built by hand, but reconstructed from noisy measurements. In `01/15_tomography_of_a_noisy_qubit` we run the three circuits of `01/11` §4 on a simulator carrying a real device's noise, rebuild $\rho$ from the three averages (the recipe of `01/10` §3), place it in the ball of section 1, and read its purity and its fidelity to the state we meant to prepare — with today's convention.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §9.2 (trace distance, the Helstrom-type interpretation, fidelity, the relations between them — note: their "fidelity" is the root $F_U = \sqrt F$ of this course's $F$), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- R. Jozsa, *Fidelity for mixed quantum states*, J. Mod. Opt. **41**, 2315–2323 (1994) — the squared convention used here, and its axioms. doi:10.1080/09500349414552171
- A. Uhlmann, *The "transition probability" in the state space of a *-algebra*, Rep. Math. Phys. **9**, 273–279 (1976) — his *transition probability* is the squared $F$ used here; the root $F_U$ carries his name by usage (as in `02/12`). doi:10.1016/0034-4877(76)90060-4
- C. A. Fuchs & J. van de Graaf, *Cryptographic distinguishability measures for quantum-mechanical states*, IEEE Trans. Inf. Theory **45**, 1216–1227 (1999) — the inequalities $1 - \sqrt F \le T \le \sqrt{1 - F}$. doi:10.1109/18.761271
- C. W. Helstrom, *Quantum Detection and Estimation Theory*, Academic Press (1976) — the optimal one-shot discrimination bound $\tfrac12(1 + T)$.
- M. Hübner, *Explicit computation of the Bures distance for density matrices*, Phys. Lett. A **163**, 239–242 (1992) — the qubit closed form $F = \mathrm{tr}(\rho\sigma) + 2\sqrt{\det\rho\det\sigma}$. doi:10.1016/0375-9601(92)91004-B

**Previous:** `notebooks/01_QuantumFoundations/13_pure_vs_mixed.ipynb` · **Next:** `notebooks/01_QuantumFoundations/15_tomography_of_a_noisy_qubit.ipynb`